In [0]:
from pyspark.sql import functions as F

print("Silver transformation started")

silver_tables = [
    "customers",
    "customer_addresses",
    "customer_contacts",
    "mobile_plans",
    "service_types",
    "plan_services",
    "subscriptions",
    "subscription_services",
    "call_records",
    "sms_records",
    "data_usage",
    "bills",
    "bill_items",
    "payments",
    "complaint_categories",
    "complaints",
    "service_areas"
]

print(f"Tables planned for Silver: {len(silver_tables)}")

In [0]:
# ============================================================
# SILVER — CUSTOMERS
# ============================================================

customers_bronze = spark.table("telecom.bronze.customers")

customers_silver = customers_bronze \
    .withColumn("date_of_birth", F.to_date("date_of_birth")) \
    .withColumn("registration_date", F.to_timestamp("registration_date")) \
    .withColumn("created_at", F.to_timestamp("created_at")) \
    .withColumn("updated_at", F.to_timestamp("updated_at")) \
    .withColumn("customer_status", F.upper(F.trim(F.col("customer_status")))) \
    .withColumn("gender", F.initcap(F.trim(F.col("gender")))) \
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name")))) \
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name")))) \
    .withColumn("email", F.lower(F.trim(F.col("email")))) \
    .dropDuplicates(["customer_id"]) \
    .filter(F.col("customer_id").isNotNull()) \
    .drop(
        "_source_table",
        "_source_file"
    )

# Write to Silver
(
    customers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.customers")
)

print(
    f"✅ Silver customers created: "
    f"{customers_silver.count():,} records"
)

display(customers_silver.limit(10))

In [0]:
# ============================================================
# SILVER — CUSTOMER ADDRESSES
# ============================================================

addresses_bronze = spark.table("telecom.bronze.customer_addresses")

addresses_silver = addresses_bronze \
    .withColumn("address_id", F.col("address_id").cast("long")) \
    .withColumn("is_primary", F.col("is_primary").cast("boolean")) \
    .withColumn("created_at", F.to_timestamp("created_at")) \
    .withColumn("address_type", F.upper(F.trim(F.col("address_type")))) \
    .withColumn("street_address", F.trim(F.col("street_address"))) \
    .withColumn("city", F.initcap(F.trim(F.col("city")))) \
    .withColumn("state", F.initcap(F.trim(F.col("state")))) \
    .withColumn("postal_code", F.trim(F.col("postal_code"))) \
    .withColumn("country", F.upper(F.trim(F.col("country")))) \
    .dropDuplicates(["address_id"]) \
    .filter(
        F.col("address_id").isNotNull() &
        F.col("customer_id").isNotNull()
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    addresses_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.customer_addresses")
)

print(
    f"✅ Silver customer_addresses created: "
    f"{addresses_silver.count():,} records"
)

display(addresses_silver.limit(10))

In [0]:
# ============================================================
# SILVER — CUSTOMER CONTACTS
# ============================================================

contacts_bronze = spark.table("telecom.bronze.customer_contacts")

contacts_silver = contacts_bronze \
    .withColumn("contact_id", F.col("contact_id").cast("long")) \
    .withColumn("created_at", F.to_timestamp("created_at")) \
    .withColumn("contact_type", F.upper(F.trim(F.col("contact_type")))) \
    .withColumn("contact_value", F.trim(F.col("contact_value"))) \
    .withColumn("is_primary", F.col("is_primary").cast("boolean")) \
    .dropDuplicates(["contact_id"]) \
    .filter(
        F.col("contact_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("contact_value").isNotNull()
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    contacts_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.customer_contacts")
)

print(
    f"✅ Silver customer_contacts created: "
    f"{contacts_silver.count():,} records"
)

display(contacts_silver.limit(10))

In [0]:
# ============================================================
# SILVER — MOBILE PLANS
# ============================================================

plans_bronze = spark.table("telecom.bronze.mobile_plans")

plans_silver = plans_bronze \
    .withColumn("monthly_charge", F.col("monthly_charge").cast("decimal(12,2)")) \
    .withColumn("plan_name", F.initcap(F.trim(F.col("plan_name")))) \
    .withColumn("plan_type", F.upper(F.trim(F.col("plan_type")))) \
    .withColumn("plan_status", F.upper(F.trim(F.col("plan_status")))) \
    .withColumn("created_at", F.to_timestamp("created_at")) \
    .withColumn("updated_at", F.to_timestamp("updated_at")) \
    .dropDuplicates(["plan_id"]) \
    .filter(
        F.col("plan_id").isNotNull() &
        F.col("monthly_charge").isNotNull() &
        (F.col("monthly_charge") >= 0)
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    plans_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.mobile_plans")
)

print(
    f"✅ Silver mobile_plans created: "
    f"{plans_silver.count():,} records"
)

display(plans_silver)

In [0]:
# ============================================================
# SILVER — SERVICE TYPES
# ============================================================

services_bronze = spark.table("telecom.bronze.service_types")

services_silver = services_bronze \
    .withColumn("service_name", F.initcap(F.trim(F.col("service_name")))) \
    .withColumn("service_category", F.upper(F.trim(F.col("service_category")))) \
    .withColumn("description", F.trim(F.col("description"))) \
    .withColumn("created_at", F.to_timestamp(F.col("created_at"))) \
    .dropDuplicates(["service_id"]) \
    .filter(
        F.col("service_id").isNotNull() &
        F.col("service_name").isNotNull()
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    services_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.service_types")
)

print(
    f"✅ Silver service_types created: "
    f"{services_silver.count():,} records"
)

display(services_silver)

In [0]:
# ============================================================
# SILVER — PLAN SERVICES
# ============================================================

plan_services_bronze = spark.table("telecom.bronze.plan_services")

plan_services_silver = plan_services_bronze \
    .withColumn(
        "plan_service_id",
        F.col("plan_service_id").cast("long")
    ) \
    .withColumn(
        "service_limit",
        F.trim(F.col("service_limit"))
    ) \
    .withColumn(
        "overage_charge",
        F.col("overage_charge").cast("decimal(10,2)")
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp(F.col("created_at"))
    ) \
    .dropDuplicates(["plan_service_id"]) \
    .filter(
        F.col("plan_service_id").isNotNull() &
        F.col("plan_id").isNotNull() &
        F.col("service_id").isNotNull()
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    plan_services_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.plan_services")
)

print(
    f"✅ Silver plan_services created: "
    f"{plan_services_silver.count():,} records"
)

display(plan_services_silver)

In [0]:
# ============================================================
# SILVER — SUBSCRIPTIONS
# ============================================================

subscriptions_bronze = spark.table("telecom.bronze.subscriptions")

subscriptions_silver = subscriptions_bronze \
    .withColumn(
        "activation_date",
        F.to_timestamp("activation_date")
    ) \
    .withColumn(
        "deactivation_date",
        F.to_timestamp("deactivation_date")
    ) \
    .withColumn(
        "subscription_status",
        F.upper(F.trim(F.col("subscription_status")))
    ) \
    .withColumn(
        "phone_number",
        F.trim(F.col("phone_number"))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at")
    ) \
    .dropDuplicates(["subscription_id"]) \
    .filter(
        F.col("subscription_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("plan_id").isNotNull() &
        F.col("phone_number").isNotNull() &
        F.col("activation_date").isNotNull()
    ) \
    .filter(
        F.col("deactivation_date").isNull() |
        (F.col("deactivation_date") >= F.col("activation_date"))
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    subscriptions_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.subscriptions")
)

print(
    f"✅ Silver subscriptions created: "
    f"{subscriptions_silver.count():,} records"
)

display(subscriptions_silver.limit(10))

In [0]:
# ============================================================
# SILVER — SUBSCRIPTION SERVICES
# ============================================================

subscription_services_bronze = spark.table(
    "telecom.bronze.subscription_services"
)

subscription_services_silver = subscription_services_bronze \
    .withColumn(
        "subscription_service_id",
        F.col("subscription_service_id").cast("long")
    ) \
    .withColumn(
        "activation_date",
        F.to_timestamp("activation_date")
    ) \
    .withColumn(
        "deactivation_date",
        F.to_timestamp("deactivation_date")
    ) \
    .withColumn(
        "service_status",
        F.upper(F.trim(F.col("service_status")))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .dropDuplicates(["subscription_service_id"]) \
    .filter(
        F.col("subscription_service_id").isNotNull() &
        F.col("subscription_id").isNotNull() &
        F.col("service_id").isNotNull()
    ) \
    .filter(
        F.col("deactivation_date").isNull() |
        (
            F.col("deactivation_date") >=
            F.col("activation_date")
        )
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    subscription_services_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.subscription_services")
)

print(
    f"✅ Silver subscription_services created: "
    f"{subscription_services_silver.count():,} records"
)

display(subscription_services_silver.limit(10))

In [0]:
# ============================================================
# SILVER — CALL RECORDS
# ============================================================

calls_bronze = spark.table("telecom.bronze.call_records")

calls_silver = calls_bronze \
    .withColumn(
        "call_start_time",
        F.to_timestamp("call_start_time")
    ) \
    .withColumn(
        "call_end_time",
        F.to_timestamp("call_end_time")
    ) \
    .withColumn(
        "call_duration_seconds",
        F.col("call_duration_seconds").cast("long")
    ) \
    .withColumn(
        "call_charges",
        F.col("call_charges").cast("decimal(12,4)")
    ) \
    .withColumn(
        "call_type",
        F.upper(F.trim(F.col("call_type")))
    ) \
    .withColumn(
        "call_direction",
        F.upper(F.trim(F.col("call_direction")))
    ) \
    .withColumn(
        "destination_number",
        F.trim(F.col("destination_number"))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .dropDuplicates(["call_id"]) \
    .filter(
        F.col("call_id").isNotNull() &
        F.col("subscription_id").isNotNull() &
        F.col("call_start_time").isNotNull() &
        F.col("call_end_time").isNotNull()
    ) \
    .filter(
        (F.col("call_duration_seconds") >= 0) &
        (F.col("call_end_time") >= F.col("call_start_time"))
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    calls_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.call_records")
)

print(
    f"✅ Silver call_records created: "
    f"{calls_silver.count():,} records"
)

display(calls_silver.limit(10))

In [0]:
# ============================================================
# SILVER — SMS RECORDS
# ============================================================

sms_bronze = spark.table("telecom.bronze.sms_records")

sms_silver = sms_bronze \
    .withColumn(
        "sms_timestamp",
        F.to_timestamp("sms_timestamp")
    ) \
    .withColumn(
        "sms_count",
        F.col("sms_count").cast("long")
    ) \
    .withColumn(
        "sms_charges",
        F.col("sms_charges").cast("decimal(12,4)")
    ) \
    .withColumn(
        "sms_type",
        F.upper(F.trim(F.col("sms_type")))
    ) \
    .withColumn(
        "sms_direction",
        F.upper(F.trim(F.col("sms_direction")))
    ) \
    .withColumn(
        "destination_number",
        F.trim(F.col("destination_number"))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .dropDuplicates(["sms_id"]) \
    .filter(
        F.col("sms_id").isNotNull() &
        F.col("subscription_id").isNotNull() &
        F.col("sms_timestamp").isNotNull()
    ) \
    .filter(
        F.col("sms_count") > 0
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    sms_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.sms_records")
)

print(
    f"✅ Silver sms_records created: "
    f"{sms_silver.count():,} records"
)

display(sms_silver.limit(10))

In [0]:
# ============================================================
# SILVER — DATA USAGE
# ============================================================

data_usage_bronze = spark.table("telecom.bronze.data_usage")

data_usage_silver = data_usage_bronze \
    .withColumn(
        "usage_date",
        F.to_date("usage_date")
    ) \
    .withColumn(
        "usage_start_time",
        F.to_timestamp("usage_start_time")
    ) \
    .withColumn(
        "usage_end_time",
        F.to_timestamp("usage_end_time")
    ) \
    .withColumn(
        "data_consumed_mb",
        F.col("data_consumed_mb").cast("decimal(14,4)")
    ) \
    .withColumn(
        "data_charges",
        F.col("data_charges").cast("decimal(12,4)")
    ) \
    .withColumn(
        "network_type",
        F.upper(F.trim(F.col("network_type")))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .dropDuplicates(["usage_id"]) \
    .filter(
        F.col("usage_id").isNotNull() &
        F.col("subscription_id").isNotNull() &
        F.col("usage_date").isNotNull() &
        F.col("usage_start_time").isNotNull() &
        F.col("usage_end_time").isNotNull()
    ) \
    .filter(
        (F.col("data_consumed_mb") >= 0) &
        (F.col("usage_end_time") >= F.col("usage_start_time"))
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    data_usage_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.data_usage")
)

print(
    f"✅ Silver data_usage created: "
    f"{data_usage_silver.count():,} records"
)

display(data_usage_silver.limit(10))

In [0]:
# ============================================================
# SILVER — BILLS
# ============================================================

bills_bronze = spark.table("telecom.bronze.bills")

bills_silver = bills_bronze \
    .withColumn(
        "bill_date",
        F.to_date("bill_date")
    ) \
    .withColumn(
        "billing_period_start",
        F.to_date("billing_period_start")
    ) \
    .withColumn(
        "billing_period_end",
        F.to_date("billing_period_end")
    ) \
    .withColumn(
        "due_date",
        F.to_date("due_date")
    ) \
    .withColumn(
        "total_amount",
        F.col("total_amount").cast("decimal(14,2)")
    ) \
    .withColumn(
        "tax_amount",
        F.col("tax_amount").cast("decimal(14,2)")
    ) \
    .withColumn(
        "discount_amount",
        F.col("discount_amount").cast("decimal(14,2)")
    ) \
    .withColumn(
        "net_amount",
        F.col("net_amount").cast("decimal(14,2)")
    ) \
    .withColumn(
        "bill_status",
        F.upper(F.trim(F.col("bill_status")))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .dropDuplicates(["bill_id"]) \
    .filter(
        F.col("bill_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("subscription_id").isNotNull() &
        F.col("bill_date").isNotNull()
    ) \
    .filter(
        (F.col("total_amount") >= 0) &
        (F.col("tax_amount") >= 0) &
        (F.col("discount_amount") >= 0) &
        (F.col("net_amount") >= 0)
    ) \
    .filter(
        F.col("billing_period_end") >=
        F.col("billing_period_start")
    ) \
    .filter(
        F.col("bill_date") >=
        F.col("billing_period_start")
    ) \
    .filter(
        F.col("due_date") >=
        F.col("bill_date")
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    bills_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.bills")
)

print(
    f"✅ Silver bills created: "
    f"{bills_silver.count():,} records"
)

display(bills_silver.limit(10))

In [0]:
# ============================================================
# SILVER — BILL ITEMS
# ============================================================

bill_items_bronze = spark.table("telecom.bronze.bill_items")

bill_items_silver = bill_items_bronze \
    .withColumn(
        "bill_item_id",
        F.col("bill_item_id").cast("long")
    ) \
    .withColumn(
        "quantity",
        F.col("quantity").cast("decimal(14,2)")
    ) \
    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(14,4)")
    ) \
    .withColumn(
        "amount",
        F.col("amount").cast("decimal(14,2)")
    ) \
    .withColumn(
        "item_type",
        F.upper(F.trim(F.col("item_type")))
    ) \
    .withColumn(
        "description",
        F.trim(F.col("description"))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .dropDuplicates(["bill_item_id"]) \
    .filter(
        F.col("bill_item_id").isNotNull() &
        F.col("bill_id").isNotNull() &
        F.col("quantity").isNotNull() &
        F.col("unit_price").isNotNull() &
        F.col("amount").isNotNull()
    ) \
    .filter(
        (F.col("quantity") > 0) &
        (F.col("unit_price") >= 0) &
        (F.col("amount") >= 0)
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    bill_items_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.bill_items")
)

print(
    f"✅ Silver bill_items created: "
    f"{bill_items_silver.count():,} records"
)

display(bill_items_silver.limit(10))

In [0]:
# ============================================================
# SILVER — PAYMENTS
# ============================================================

payments_bronze = spark.table("telecom.bronze.payments")

payments_silver = payments_bronze \
    .withColumn(
        "payment_date",
        F.to_timestamp("payment_date")
    ) \
    .withColumn(
        "payment_amount",
        F.col("payment_amount").cast("decimal(14,2)")
    ) \
    .withColumn(
        "payment_method",
        F.upper(F.trim(F.col("payment_method")))
    ) \
    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status")))
    ) \
    .withColumn(
        "transaction_reference",
        F.trim(F.col("transaction_reference"))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    ) \
    .dropDuplicates(["payment_id"]) \
    .filter(
        F.col("payment_id").isNotNull() &
        F.col("bill_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("payment_date").isNotNull()
    ) \
    .filter(
        F.col("payment_amount") > 0
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    payments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.payments")
)

print(
    f"✅ Silver payments created: "
    f"{payments_silver.count():,} records"
)

display(payments_silver.limit(10))

In [0]:
# ============================================================
# SILVER — COMPLAINT CATEGORIES
# ============================================================

categories_bronze = spark.table(
    "telecom.bronze.complaint_categories"
)

categories_silver = categories_bronze \
    .withColumn(
        "category_name",
        F.initcap(F.trim(F.col("category_name")))
    ) \
    .withColumn(
        "description",
        F.trim(F.col("description"))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp(F.col("created_at"))
    ) \
    .dropDuplicates(["category_id"]) \
    .filter(
        F.col("category_id").isNotNull() &
        F.col("category_name").isNotNull()
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    categories_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.complaint_categories")
)

print(
    f"✅ Silver complaint_categories created: "
    f"{categories_silver.count():,} records"
)

display(categories_silver)

In [0]:
# ============================================================
# SILVER — COMPLAINTS
# ============================================================

complaints_bronze = spark.table(
    "telecom.bronze.complaints"
)

complaints_silver = complaints_bronze \
    .withColumn(
        "complaint_date",
        F.to_timestamp("complaint_date")
    ) \
    .withColumn(
        "resolution_date",
        F.to_timestamp("resolution_date")
    ) \
    .withColumn(
        "complaint_status",
        F.upper(F.trim(F.col("complaint_status")))
    ) \
    .withColumn(
        "priority",
        F.upper(F.trim(F.col("priority")))
    ) \
    .withColumn(
        "complaint_description",
        F.trim(F.col("complaint_description"))
    ) \
    .withColumn(
        "resolution_notes",
        F.trim(F.col("resolution_notes"))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp(F.col("created_at"))
    ) \
    .withColumn(
        "updated_at",
        F.to_timestamp(F.col("updated_at"))
    ) \
    .dropDuplicates(["complaint_id"]) \
    .filter(
        F.col("complaint_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("category_id").isNotNull() &
        F.col("complaint_date").isNotNull()
    ) \
    .filter(
        F.col("resolution_date").isNull() |
        (
            F.col("resolution_date") >=
            F.col("complaint_date")
        )
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    complaints_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.complaints")
)

print(
    f"✅ Silver complaints created: "
    f"{complaints_silver.count():,} records"
)

display(complaints_silver.limit(10))

In [0]:
# ============================================================
# SILVER — SERVICE AREAS
# ============================================================

service_areas_bronze = spark.table(
    "telecom.bronze.service_areas"
)

service_areas_silver = service_areas_bronze \
    .withColumn(
        "area_name",
        F.trim(F.col("area_name"))
    ) \
    .withColumn(
        "city",
        F.initcap(F.trim(F.col("city")))
    ) \
    .withColumn(
        "state",
        F.initcap(F.trim(F.col("state")))
    ) \
    .withColumn(
        "region",
        F.initcap(F.trim(F.col("region")))
    ) \
    .withColumn(
        "network_coverage",
        F.upper(F.trim(F.col("network_coverage")))
    ) \
    .withColumn(
        "created_at",
        F.to_timestamp(F.col("created_at"))
    ) \
    .dropDuplicates(["area_id"]) \
    .filter(
        F.col("area_id").isNotNull() &
        F.col("area_name").isNotNull() &
        F.col("city").isNotNull() &
        F.col("state").isNotNull()
    ) \
    .drop(
        "_source_table",
        "_source_file"
    )

(
    service_areas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.silver.service_areas")
)

print(
    f"✅ Silver service_areas created: "
    f"{service_areas_silver.count():,} records"
)

display(service_areas_silver.limit(10))

In [0]:
# ============================================================
# SILVER LAYER — FINAL VALIDATION
# ============================================================

expected_counts = {
    "customers": 1000,
    "customer_addresses": 1278,
    "customer_contacts": 2000,
    "mobile_plans": 5,
    "service_types": 10,
    "plan_services": 25,
    "subscriptions": 1242,
    "subscription_services": 6232,
    "call_records": 50000,
    "sms_records": 100000,
    "data_usage": 75000,
    "bills": 5039,
    "bill_items": 15117,
    "payments": 4074,
    "complaint_categories": 7,
    "complaints": 500,
    "service_areas": 50
}

silver_validation = []

for table_name, expected_count in expected_counts.items():

    table_name_full = f"telecom.silver.{table_name}"

    try:
        actual_count = spark.table(table_name_full).count()

        silver_validation.append({
            "table_name": table_name,
            "expected_records": expected_count,
            "actual_records": actual_count,
            "status": "PASS"
            if actual_count == expected_count
            else "CHECK"
        })

    except Exception as e:

        silver_validation.append({
            "table_name": table_name,
            "expected_records": expected_count,
            "actual_records": -1,
            "status": "MISSING"
        })

silver_validation_df = spark.createDataFrame(silver_validation)

display(
    silver_validation_df.orderBy("table_name")
)

print("=" * 70)
print("SILVER LAYER VALIDATION")
print("=" * 70)

total_tables = len(silver_validation)
passed_tables = sum(
    1 for row in silver_validation
    if row["status"] == "PASS"
)

print(f"Expected tables : {total_tables}")
print(f"Passed tables   : {passed_tables}")
print("=" * 70)